# SPAN 2 Methodology and Functionality

https://www.cmegroup.com/clearing/files/cme-span-2-margin-framework.pdf

Framework Components and Risk Methodology:
Total Portfolio Margin= x*Historical Risk + (1-x)*Stress Risk + Liquidity + Concentration

A simple example to illustrate the general concept of margin model framework with a single stock only.

## Setup

Long 200 futures contracts at settlement price 4000 (this is the price at the end of the day, assumed price for start of next day), where each contract is $20 per point

## Historical Risk

Idea is to look at the daily changes over the last 10 years, and then calculate the VaR value for it.

Also need to rescale these values by taking into account factors like correlation and volatility.

For this example, only worst risk as historical risk and use the volatility scaled returns.

In [ ]:
import pandas as pd
market_price = 4000
position = 200
per_point_value = 20
market_volatility = 0.15
days_table = pd.DataFrame({'Past_day': ['H1','H2','H3','H4'], 'Past_day_open': [1000, 2000, 1600, 2500],
                           'Past_day_close': [1030, 1950, 1648, 2450], 'Vol_estimate':[0.15, 0.3, 0.1,0.2]})
print(days_table)

  Past_day  Previous_settlement  Settlement  Vol_estimate
0        P                 1000        1030          0.15
1        Q                 2000        1950          0.30
2        R                 1600        1648          0.10
3        S                 2500        2450          0.20


In [ ]:
import numpy as np
days_table['Return'] = np.round((days_table['Past_day_close']-days_table['Past_day_open'])/days_table['Past_day_open'],2)
days_table['Adjusted_return'] = days_table['Return']*market_volatility/days_table['Vol_estimate']
days_table['Adjusted_profit_loss'] =-market_price*position*per_point_value*days_table['Adjusted_return']
historical_risk = np.round(max(days_table['Adjusted_profit_loss']),2)
print(days_table)
print(f"Historical Risk: {historical_risk}")


  Past_day  Previous_settlement  Settlement  Vol_estimate  return  \
0        P                 1000        1030          0.15    0.03   
1        Q                 2000        1950          0.30   -0.02   
2        R                 1600        1648          0.10    0.03   
3        S                 2500        2450          0.20   -0.02   

   profit_loss  Return  Profit_loss  Adjusted_return  Adjusted_profit_loss  
0       2400.0    0.03       2400.0            0.030                2400.0  
1      -1600.0   -0.02      -1600.0           -0.010                -800.0  
2       2400.0    0.03       2400.0            0.045                3600.0  
3      -1600.0   -0.02      -1600.0           -0.015               -1200.0  
3600.0
2400.0


In [7]:
print(days_table)

  Past_day  Previous_settlement  Settlement  Vol_estimate  return  \
0        P                 1000        1030          0.15    0.03   
1        Q                 2000        1950          0.30   -0.02   
2        R                 1600        1648          0.10    0.03   
3        S                 2500        2450          0.20   -0.02   

   profit_loss  Return  Profit_loss  Adjusted_return  Adjusted_profit_loss  
0       2400.0    0.03       2400.0            0.030                2400.0  
1      -1600.0   -0.02      -1600.0           -0.010                -800.0  
2       2400.0    0.03       2400.0            0.045                3600.0  
3      -1600.0   -0.02      -1600.0           -0.015               -1200.0  


## Stress Risk

For stress risk, the idea is to use the raw results from extreme periods, not rescaling by volatility or concentration.

This basically checks for an overcorrection from the historical risk.

In [ ]:
stress_table = pd.DataFrame({'Scenario': ['E1','E2','E3'], 'Return':[-0.07, 0.05,-0.03], 'Scenario_price':[3560, 3600, 4340] })
stress_table['Return'] = (stress_table['Scenario_price']-market_price)/market_price
stress_table['Profit_Loss'] = market_price*position*per_point_value*stress_table['Return']
stress_risk = np.round(max(stress_table['Profit_loss']),2)
print(f"Stress Risk: {stress_risk}")

## Liquidity and Concentration

Liquidity is the money required to close the contracts at the top price

The idea of concentration is to account for the fact that for a large number of orders, you need to go beyond the initial bid or ask value because the size at that value may be smaller than the number of orders.

The instructions for calculation of this are vague on the website. Books like John C Hull have the idea of 
Liquidity Cost = (1/2)*(mean_spread + z*std_spread)*Position_value

In practice, you would use mutliple spreads from different time periods or take an average.

Here, only one  table with spread_sizes and size

In [ ]:
spread_table = pd.DataFrame({'spread':[1, 1.5, 2, 2.5, 3.5], 'bid_size': [40, 60, 100, 150, 150]})
spread_table['spread'].sort_values(ascending=False)
print(spread_table)


   bid_price  bid_size
0     3999.5        40
1     3998.5        60
2     3997.0       100
3     3995.0       150
4     3992.0       150


In [ ]:
concentration = 0
if np.sum(spread_table['bid_size'])<position:
    print(f"Not enough contracts available to close position")
else:
    for i in range(len(spread_table)):
        take_size = min(position, spread_table['bid_size'].iloc[i])
        concentration += spread_table['spread'].iloc[i]*take_size
        position -= take_size
        if position<=0:
            break
print(f"Concentration: {concentration}")
        